# Module 3: Statistical Inference in Modelling

This module transitions from descriptive statistics to **Inferential Statistics**, the formal framework for making rigorous, data-driven conclusions about a population parameter ($\theta$) using data from a limited sample ($n$).

## 1. Core Objectives

- **Interval Estimation**: Construct confidence intervals for population means under known and unknown variance conditions.
- **Sample Size Determination**: Calculate the minimum sample size ($n$) required to achieve a specific precision.
- **Hypothesis Testing**: Formulate and execute formal frameworks to decide between competing claims ($H_0$ vs. $H_a$).

## 2. Interval Estimation (Confidence Intervals)

Rather than a point estimate (a "spear"), we use a "net" to capture the true parameter with a defined level of confidence.

### Known Population Variance ($\sigma$)
Use the Standard Normal distribution ($Z$).

$$ CI = \bar{x} \pm Z_{\alpha/2} \left( \frac{\sigma}{\sqrt{n}} \right) $$

In [ ]:
import numpy as np
from scipy import stats

# Example: Known Variance
sample_mean = 14200
pop_std = 500  # Known population standard deviation
n = 30
confidence_level = 0.95

# Z critical value
z_critical = stats.norm.ppf((1 + confidence_level) / 2)
margin_of_error = z_critical * (pop_std / np.sqrt(n))

ci_lower = sample_mean - margin_of_error
ci_upper = sample_mean + margin_of_error

print(f"{confidence_level*100}% Confidence Interval (Known Variance): [{ci_lower:.2f}, {ci_upper:.2f}]")

### Unknown Population Variance ($s$)
Use the Student’s $t$-distribution.

$$ CI = \bar{x} \pm t_{\alpha/2, \nu} \left( \frac{s}{\sqrt{n}} \right) $$

Where $\nu = n - 1$ degrees of freedom.

In [ ]:
import numpy as np
from scipy import stats

# Example: Calculating a 95% Confidence Interval (Unknown Variance)
data = [14200, 13800, 14500, 14100, 14300] # Sample data
n = len(data)
mean = np.mean(data)
std_err = stats.sem(data) # Standard error of the mean

# 95% confidence interval using t-distribution
ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=std_err)
print(f"Confidence Interval (Unknown Variance): {ci}")

## 3. Sample Size Determination

To achieve a desired margin of error ($E$), we can rearrange the CI formula to solve for $n$:

$$ n = \left( \frac{Z_{\alpha/2} \cdot \sigma}{E} \right)^2 $$

In [ ]:
# Example: Minimum Sample Size Calculation
desired_margin_of_error = 100
estimated_pop_std = 500
confidence_level = 0.95
z_critical = stats.norm.ppf((1 + confidence_level) / 2)

n_required = (z_critical * estimated_pop_std / desired_margin_of_error) ** 2
print(f"Minimum sample size required: {int(np.ceil(n_required))}") # Always round up

## 4. Hypothesis Testing Framework

A formal courtroom-like procedure to evaluate status quo claims.

- **Null Hypothesis ($H_0$)**: The "no effect" baseline.
- **Alternative Hypothesis ($H_a$)**: The effect you are attempting to prove.
- **Significance Level ($\alpha$)**: The threshold for Type I Error (False Positive) risk (commonly 0.05).

In [ ]:
# Setting up Hypothesis Testing Parameters
alpha = 0.05
null_hypothesis = "The population mean is equal to 14000"
alt_hypothesis = "The population mean is not equal to 14000"

print(f"Significance Level (alpha): {alpha}")
print(f"H0: {null_hypothesis}")
print(f"Ha: {alt_hypothesis}")

### Test Statistic
Standardizes the difference relative to noise.

$$ Z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}} $$

In [ ]:
# 1-Sample Z-Test (Known Variance)
sample_data = [14200, 13800, 14500, 14100, 14300, 14600, 13900, 14400]
mu_0 = 14000
pop_std = 400

z_stat, p_value = stats.ztest(sample_data, value=mu_0, alternative='two-sided')
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Reject H0: {p_value < alpha}")

In [ ]:
# 1-Sample T-Test (Unknown Variance)
t_stat, p_value_t = stats.ttest_1samp(sample_data, popmean=mu_0)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value_t:.4f}")
print(f"Reject H0: {p_value_t < alpha}")

### Two-Sample Hypothesis Testing (Independent)
Comparing the means of two independent groups.

In [ ]:
# Independent Two-Sample T-Test
group_a = [14200, 13800, 14500, 14100, 14300]
group_b = [15000, 14800, 15200, 14900, 15100]

t_stat_ind, p_value_ind = stats.ttest_ind(group_a, group_b)
print(f"Independent T-statistic: {t_stat_ind:.4f}")
print(f"P-value: {p_value_ind:.4f}")
print(f"Reject H0 (Means are different): {p_value_ind < alpha}")

### Paired Hypothesis Testing
Comparing the means of the same group at two different times (e.g., before and after).

In [ ]:
# Paired T-Test
before = [14200, 13800, 14500, 14100, 14300]
after = [14500, 14100, 14800, 14400, 14600] # e.g., after a training program

t_stat_paired, p_value_paired = stats.ttest_rel(before, after)
print(f"Paired T-statistic: {t_stat_paired:.4f}")
print(f"P-value: {p_value_paired:.4f}")
print(f"Reject H0 (Significant difference): {p_value_paired < alpha}")

## 5. Summary of Key Constraints & Traps

### The Probability Fallacy & Practical Significance
- **Probability Fallacy**: A 95% CI does not mean there is a 95% probability the true parameter is inside your specific interval; it means the *procedure* captures the true parameter 95% of the time in the long run.
- **Statistical vs. Practical Significance**: With massive sample sizes ($n \to \infty$), even trivial differences become statistically significant ($p < 0.05$). Always evaluate **Effect Size** (e.g., Cohen's $d$) for business impact.

In [ ]:
# Calculating Effect Size (Cohen's d) for Practical Significance
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std

d = cohens_d(group_a, group_b)
print(f"Cohen's d: {d:.4f}")
print("Interpretation: 0.2=small, 0.5=medium, 0.8=large effect")

### Type I vs. Type II Errors & Power
- **Type I ($\alpha$)**: Rejecting $H_0$ when it is true (False Positive).
- **Type II ($\beta$)**: Failing to reject $H_0$ when it is false (False Negative).
- **Power ($1 - \beta$)**: The probability of correctly rejecting a false $H_0$.

In [ ]:
# Calculating Statistical Power using statsmodels
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()
effect_size = abs(d) # Use absolute value of Cohen's d
alpha = 0.05
nobs = 50 # Sample size per group

power = analysis.power(effect_size=effect_size, nobs1=nobs, alpha=alpha, ratio=1.0)
print(f"Statistical Power: {power:.4f}")
print(f"Type II Error Rate (Beta): {1 - power:.4f}")

## 6. Visualizing Inference

Visualizing the $t$-distribution, critical regions, and our test statistic helps build intuition about hypothesis testing.

In [ ]:
import matplotlib.pyplot as plt

# Plot T-distribution with Critical Regions
df = n - 1
x = np.linspace(-4, 4, 1000)
y = stats.t.pdf(x, df)

plt.figure(figsize=(10, 6))
plt.plot(x, y, 'b-', lw=2, label=f't-distribution (df={df})')

# Critical values for alpha = 0.05 (two-tailed)
t_crit = stats.t.ppf(1 - alpha/2, df)
plt.fill_between(x[x <= -t_crit], y[x <= -t_crit], color='red', alpha=0.3, label='Rejection Region (α/2)')
plt.fill_between(x[x >= t_crit], y[x >= t_crit], color='red', alpha= is a visual representation of the confidence interval calculated earlier.

In [ ]:
# Visualizing the Confidence Interval
plt.figure(figsize=(8, 4))
plt.errorbar(x=1, y=mean, yerr=margin_of_error, fmt='o', color='blue', capsize=5, 
             label=f'95% CI: [{ci[0]:.0f}, {ci[1]:.0f}]')
plt.axhline(y=mu_0, color='red', linestyle='--', label=f'Null Hypothesis Mean ({mu_0})')
plt.xlim(0.5, 1.5)
plt.xticks([1], ['Sample Mean'])
plt.ylabel('Value')
plt.title('Confidence Interval Visualization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Conclusion

Statistical inference provides the mathematical rigor needed to move from observing data to making actionable, population-level decisions. Always remember to:
1. Check assumptions (normality, independence, variance).
2. Report both statistical significance ($p$-value) and practical significance (Effect Size).
3. Understand the trade-offs between Type I and Type II errors in your specific business context.